### Importação das bibliotecas

In [208]:
import pandas as pd

### Carregamento dos dados

In [209]:
df = pd.read_json("../data/raw/custos_importacao.json")
df.head(10)

,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."
5,6,Transponder AIS Vector,eletrônicos,"[{'start_date': '23/07/2020', 'usd_price': 228..."
6,7,Radar AIS Zen,eletrônicos,"[{'start_date': '26/10/2016', 'usd_price': 625..."
7,8,GPS AIS Zen,eletrônicos,"[{'start_date': '06/12/2019', 'usd_price': 119..."
8,9,Transponder AIS Titan Pulse,eletrônicos,"[{'start_date': '26/12/2018', 'usd_price': 101..."
9,10,Piloto Automático Simrad Titan Flux Magnum,eletrônicos,"[{'start_date': '29/05/2018', 'usd_price': 859..."


In [210]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     150 non-null    int64 
 1   product_name   150 non-null    str   
 2   category       150 non-null    str   
 3   historic_data  150 non-null    object
dtypes: int64(1), object(1), str(2)
memory usage: 4.8+ KB


In [211]:
df.iloc[0]['historic_data']

[{'start_date': '10/08/2016', 'usd_price': 10583.63},
 {'start_date': '15/06/2018', 'usd_price': 8778.36},
 {'start_date': '25/09/2018', 'usd_price': 8023.87},
 {'start_date': '19/03/2019', 'usd_price': 8772.78},
 {'start_date': '17/01/2020', 'usd_price': 7918.18},
 {'start_date': '17/06/2020', 'usd_price': 6310.01},
 {'start_date': '02/07/2021', 'usd_price': 6586.7},
 {'start_date': '16/05/2022', 'usd_price': 6538.2},
 {'start_date': '28/02/2023', 'usd_price': 6360.91},
 {'start_date': '17/10/2023', 'usd_price': 6574.8},
 {'start_date': '16/02/2024', 'usd_price': 6657.12},
 {'start_date': '22/02/2024', 'usd_price': 6703.2},
 {'start_date': '15/03/2024', 'usd_price': 6633.66},
 {'start_date': '02/08/2024', 'usd_price': 5774.5},
 {'start_date': '08/04/2025', 'usd_price': 5579.75}]

A coluna "historic_data" armazena o histórico de preços de cada produto em um campo aninhado, o que dificulta consultas e análises no banco de dados. Para normalizar essa estrutura, cada entrada do histórico será expandida em uma linha individual, resultando em um registro por produto por data, com as colunas (product_id, product_name, category, start_date e usd_price).

### Ajuste de "historic_data"

In [212]:
df_exploded = df.explode('historic_data')
df_exploded.head(10)

,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '10/08/2016', 'usd_price': 1058..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '15/06/2018', 'usd_price': 8778..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '25/09/2018', 'usd_price': 8023..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '19/03/2019', 'usd_price': 8772..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '17/01/2020', 'usd_price': 7918..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '17/06/2020', 'usd_price': 6310..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '02/07/2021', 'usd_price': 6586.7}"
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '16/05/2022', 'usd_price': 6538.2}"
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '28/02/2023', 'usd_price': 6360..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '17/10/2023', 'usd_price': 6574.8}"


In [213]:
df_historico = pd.json_normalize(df_exploded['historic_data'])
df_historico.head(10)


,start_date,usd_price
0,10/08/2016,10583.63
0,15/06/2018,8778.36
0,25/09/2018,8023.87
0,19/03/2019,8772.78
0,17/01/2020,7918.18
0,17/06/2020,6310.01
0,02/07/2021,6586.70
0,16/05/2022,6538.20
0,28/02/2023,6360.91
0,17/10/2023,6574.80


In [214]:
df = pd.concat([df_exploded[["product_id", "product_name", "category"]].reset_index(drop=True), df_historico.reset_index(drop=True)], axis=1)
df

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,25/09/2018,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,19/03/2019,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,17/01/2020,7918.18
...,...,...,...,...,...
1255,150,Cabo de Nylon Danforth Magnum Vox,ancoragem,01/06/2023,326.88
1256,150,Cabo de Nylon Danforth Magnum Vox,ancoragem,31/01/2024,332.26
1257,150,Cabo de Nylon Danforth Magnum Vox,ancoragem,29/08/2024,292.03
1258,150,Cabo de Nylon Danforth Magnum Vox,ancoragem,04/10/2024,300.96


coluna 'historic_data' devidamente ajustada para 'start_date' e 'usd_price', agora farei algumas verificaçoes

### Verificações e ajustes

In [215]:
df.duplicated().sum()

np.int64(0)

In [216]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    1260 non-null   int64  
 1   product_name  1260 non-null   str    
 2   category      1260 non-null   str    
 3   start_date    1260 non-null   str    
 4   usd_price     1260 non-null   float64
dtypes: float64(1), int64(1), str(3)
memory usage: 49.3 KB


passarei o 'start_data' para o tipo data

In [217]:
df['start_date'] = pd.to_datetime(df['start_date'], dayfirst=True)

In [218]:
df.head(10)

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,2016-08-10,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,2018-06-15,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,2018-09-25,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,2019-03-19,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,2020-01-17,7918.18
5,1,Transponder AIS Maré Magnum,eletrônicos,2020-06-17,6310.01
6,1,Transponder AIS Maré Magnum,eletrônicos,2021-07-02,6586.70
7,1,Transponder AIS Maré Magnum,eletrônicos,2022-05-16,6538.20
8,1,Transponder AIS Maré Magnum,eletrônicos,2023-02-28,6360.91
9,1,Transponder AIS Maré Magnum,eletrônicos,2023-10-17,6574.80


In [219]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    1260 non-null   int64         
 1   product_name  1260 non-null   str           
 2   category      1260 non-null   str           
 3   start_date    1260 non-null   datetime64[us]
 4   usd_price     1260 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 49.3 KB


In [220]:
df.to_csv('../data/processed/custos_importacao_processed.csv', index=False)

In [221]:
# dff = pd.read_csv("../data/processed/custos_importacao_processed.csv", parse_dates=['start_date'])
# dff.info()